In [1]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm

In [3]:
# AAPL SANITY CHECK SCRIPT

# =========================================================
# Configuration & Whitelists
# =========================================================
source_dir = './data/walk_forward'  
dest_dir = './data/test_mmap_subset' # New folder for the subset test
TEST_TICKER = 'AAPL'

# 1. Define the years
target_years = ['2007', '2008', '2009', '2010', '2011', '2012']

# 2. Define the core features explicitly
CORE_FEATURES = [
    'm5_mom_lrsi', 'm5_mom_tsi', 'm5_vol_rel_vol', 'm5_vol_rel_cum_vol', 
    'm5_rrs_mom_5', 'd1_rrs_mom_5', 'd1_lrsi_lrsi', 'd1_adx_adx', 'd1_natr', 'dist_m5_ma_ema_8', 
    'dist_m5_ma_vwap', 'dist_m5_lvl_yest_close', 'dist_m5_lvl_hod', 'dist_m5_lvl_lod', 
    'dist_d1_trend_sma_20', 'dist_d1_trend_sma_50', 'dist_d1_trend_high_52w', 
    'spy_m5_rel_vol', 'spy_m5_chg_from_yest_close_pct', 'dist_spy_m5_vwap', 
    'dist_spy_d1_sma_20', 'target_reg_5m_logret'
]

os.makedirs(dest_dir, exist_ok=True)

# Filter files to only target years
all_files = sorted([f for f in os.listdir(source_dir) if f.endswith('.parquet')])
files = [f for f in all_files if any(year in f for year in target_years)]

if not files:
    raise FileNotFoundError(f"No parquet files found matching {target_years} in {source_dir}")

print(f"--- Starting Subset Test Run for Ticker: {TEST_TICKER} ---")

# =========================================================
# Phase 1: Data Discovery (Single Ticker)
# =========================================================
print("\nPhase 1: Scanning files for AAPL row counts...")
total_rows = 0

for file in files:
    file_path = os.path.join(source_dir, file)
    try:
        # Fast read: Only count the rows where ticker is AAPL
        df_test = pd.read_parquet(file_path, columns=['ticker'], filters=[('ticker', '==', TEST_TICKER)])
        total_rows += len(df_test)
    except Exception as e:
        print(f"Skipping {file} due to read error or missing ticker: {e}")

print(f"Total historical rows for {TEST_TICKER} (2007-2012): {total_rows}")
print(f"Features per row: {len(CORE_FEATURES)}")

if total_rows == 0:
    raise ValueError(f"Ticker {TEST_TICKER} not found in target years!")

# Save Test Metadata
metadata = {
    "feature_names": CORE_FEATURES,
    "num_features": len(CORE_FEATURES),
    "reference_ticker": TEST_TICKER,
    "price_column": 'target_reg_5m_logret',
    "years_included": target_years
}
with open(os.path.join(dest_dir, 'master_metadata_test.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

# =========================================================
# Phase 2: Pre-allocate .npy files
# =========================================================
print("\nPhase 2: Pre-allocating .npy arrays...")
feature_path = os.path.join(dest_dir, f"{TEST_TICKER}.npy")
time_path = os.path.join(dest_dir, f"{TEST_TICKER}_time.npy")

np.lib.format.open_memmap(feature_path, mode='w+', dtype=np.float32, shape=(total_rows, len(CORE_FEATURES)))
np.lib.format.open_memmap(time_path, mode='w+', dtype=np.int64, shape=(total_rows,))

# =========================================================
# Phase 3: Populate Data
# =========================================================
print("\nPhase 3: Populating data year by year...")
write_pointer = 0

# Columns we actually need to read from the disk
columns_to_read = ['ticker', 'timestamp'] + CORE_FEATURES

for file in tqdm(files, desc="Writing Data"):
    file_path = os.path.join(source_dir, file)
    
    # Highly optimized read: Only load AAPL, and ONLY load the whitelisted columns
    df_aapl = pd.read_parquet(
        file_path, 
        columns=columns_to_read, 
        filters=[('ticker', '==', TEST_TICKER)]
    )
    
    if len(df_aapl) == 0:
        continue
        
    chunk_size = len(df_aapl)
    end_idx = write_pointer + chunk_size
    
    # 1. Write Timestamps
    timestamps_array = df_aapl['timestamp'].astype('datetime64[ns]').astype(np.int64).values
    time_mmap = np.lib.format.open_memmap(time_path, mode='r+')
    time_mmap[write_pointer:end_idx] = timestamps_array
    time_mmap.flush()
    
    # 2. Write Features
    features_array = df_aapl[CORE_FEATURES].astype(np.float32).values
    feature_mmap = np.lib.format.open_memmap(feature_path, mode='r+')
    feature_mmap[write_pointer:end_idx] = features_array
    feature_mmap.flush()
    
    write_pointer = end_idx

# =========================================================
# Phase 4: Verification & Sampling
# =========================================================
print("\n" + "="*40)
print("   DATA INTEGRITY VERIFICATION   ")
print("="*40)

loaded_features = np.load(feature_path, mmap_mode='r')
loaded_times = np.load(time_path, mmap_mode='r')

print(f"Feature Array Shape: {loaded_features.shape}")
print(f"Time Array Shape:    {loaded_times.shape}")

# 1. Boundary Checks
first_time = pd.to_datetime(loaded_times[0])
last_time = pd.to_datetime(loaded_times[-1])

print(f"\n[Boundaries]")
print(f"First Timestamp: {first_time} (Expected ~2007)")
print(f"Last Timestamp:  {last_time} (Expected ~2012)")

# 2. Random Sampling Check
print(f"\n[Random Sampling]")
np.random.seed(42) 
target_col = 'target_reg_5m_logret'

for i in range(3):
    rand_idx = np.random.randint(0, total_rows)
    sample_time = pd.to_datetime(loaded_times[rand_idx])
    
    target_idx = CORE_FEATURES.index(target_col)
    sample_target = loaded_features[rand_idx, target_idx]
    sample_f1 = loaded_features[rand_idx, 0]
    
    print(f"  Row {rand_idx:<8} | Time: {sample_time} | {target_col}: {sample_target:+.6f} | {CORE_FEATURES[0]}: {sample_f1:+.4f}")

print("\nTest Script Completed Successfully!")

--- Starting Subset Test Run for Ticker: AAPL ---

Phase 1: Scanning files for AAPL row counts...
Total historical rows for AAPL (2007-2012): 119158
Features per row: 22

Phase 2: Pre-allocating .npy arrays...

Phase 3: Populating data year by year...


Writing Data: 100%|██████████| 6/6 [00:06<00:00,  1.08s/it]


   DATA INTEGRITY VERIFICATION   
Feature Array Shape: (119158, 22)
Time Array Shape:    (119158,)

[Boundaries]
First Timestamp: 2007-01-03 09:30:00 (Expected ~2007)
Last Timestamp:  2012-12-31 16:00:00 (Expected ~2012)

[Random Sampling]
  Row 15795    | Time: 2007-10-18 12:20:00 | target_reg_5m_logret: -0.000293 | m5_mom_lrsi: +0.3237
  Row 860      | Time: 2007-01-18 15:20:00 | target_reg_5m_logret: +0.001152 | m5_mom_lrsi: -1.0000
  Row 103694   | Time: 2012-03-21 09:40:00 | target_reg_5m_logret: +0.002711 | m5_mom_lrsi: +0.3294

Test Script Completed Successfully!


In [4]:
# SCRIPT FOR FULL UNIVERSE CONVERSION (CURRENTLY: 2007-2012)

import pandas as pd
import numpy as np
import os
import json
import gc
import random
from tqdm import tqdm

# =========================================================
# 1. Configuration & Whitelists
# =========================================================
source_dir = './data/walk_forward'  
dest_dir = './data/data_mmap'
os.makedirs(dest_dir, exist_ok=True)

# Define the years
target_years = ['2007', '2008', '2009', '2010', '2011', '2012']

# Define the core features explicitly (Add your newly expanded features here)
CORE_FEATURES = [
    'm5_mom_lrsi', 'm5_mom_tsi', 'm5_vol_rel_vol', 'm5_vol_rel_cum_vol', 
    'd1_lrsi_lrsi', 'd1_adx_adx', 'd1_natr', 'dist_m5_ma_ema_8', 
    'dist_m5_ma_vwap', 'dist_m5_lvl_yest_close', 'dist_m5_lvl_hod', 'dist_m5_lvl_lod', 
    'dist_d1_trend_sma_20', 'dist_d1_trend_sma_50', 'dist_d1_trend_high_52w', 
    'spy_m5_rel_vol', 'spy_m5_chg_from_yest_close_pct', 'dist_spy_m5_vwap', 
    'dist_spy_d1_sma_20', 'target_reg_5m_logret'
    # ... Add your additional core features here ...
]

# Filter files to only target years
all_files = sorted([f for f in os.listdir(source_dir) if f.endswith('.parquet')])
files = [f for f in all_files if any(year in f for year in target_years)]

if not files:
    raise FileNotFoundError(f"No parquet files found matching {target_years} in {source_dir}")

# =========================================================
# 2. Phase 1: Global Discovery (Subset Ticker Union)
# =========================================================
print(f"--- Phase 1: Scanning {len(files)} files (Years {target_years[0]} to {target_years[-1]}) ---")
ticker_row_counts = {}

for file in tqdm(files, desc="Scanning Metadata"):
    file_path = os.path.join(source_dir, file)
    # Read only the ticker column to save RAM
    df_temp = pd.read_parquet(file_path, columns=['ticker'])
    
    counts = df_temp['ticker'].value_counts().to_dict()
    for t, count in counts.items():
        ticker_row_counts[t] = ticker_row_counts.get(t, 0) + count
        
    del df_temp
    gc.collect()

print(f"\nDiscovered {len(ticker_row_counts)} unique tickers in this timeframe.")

with open(os.path.join(dest_dir, 'master_metadata.json'), 'w') as f:
    json.dump({
        "feature_names": CORE_FEATURES, 
        "price_column": 'target_reg_5m_logret',
        "total_tickers": len(ticker_row_counts),
        "years_included": target_years
    }, f, indent=4)

# =========================================================
# 3. Phase 2: Pre-allocate Subset Files
# =========================================================
print(f"\n--- Phase 2: Allocating {len(ticker_row_counts)*2} files ---")
write_pointers = {t: 0 for t in ticker_row_counts.keys()}

for t, total_rows in tqdm(ticker_row_counts.items(), desc="Allocating"):
    np.lib.format.open_memmap(os.path.join(dest_dir, f"{t}.npy"), mode='w+', 
                              dtype=np.float32, shape=(total_rows, len(CORE_FEATURES)))
    np.lib.format.open_memmap(os.path.join(dest_dir, f"{t}_time.npy"), mode='w+', 
                              dtype=np.int64, shape=(total_rows,))

# =========================================================
# 4. Phase 3: Populating Data (Only 2007-2012)
# =========================================================
print("\n--- Phase 3: Populating Subset Data ---")
columns_to_read = ['ticker', 'timestamp'] + CORE_FEATURES

for file in tqdm(files, desc="Processing Years"):
    file_path = os.path.join(source_dir, file)
    
    # Pre-filter to only hold our whitelist in RAM
    df_year = pd.read_parquet(file_path, columns=columns_to_read)
    
    for ticker_name, df_group in df_year.groupby('ticker'):
        start = write_pointers[ticker_name]
        size = len(df_group)
        end = start + size
        
        # Write Timestamps
        t_mmap = np.lib.format.open_memmap(os.path.join(dest_dir, f"{ticker_name}_time.npy"), mode='r+')
        t_mmap[start:end] = df_group['timestamp'].astype('datetime64[ns]').astype(np.int64).values
        
        # Write Features
        f_mmap = np.lib.format.open_memmap(os.path.join(dest_dir, f"{ticker_name}.npy"), mode='r+')
        f_mmap[start:end] = df_group[CORE_FEATURES].astype(np.float32).values
        
        write_pointers[ticker_name] = end
    
    # RAM cleanup
    del df_year
    gc.collect()

--- Phase 1: Scanning 6 files (Years 2007 to 2012) ---


Scanning Metadata: 100%|██████████| 6/6 [00:05<00:00,  1.09it/s]



Discovered 645 unique tickers in this timeframe.

--- Phase 2: Allocating 1290 files ---


Allocating: 100%|██████████| 645/645 [00:00<00:00, 2989.25it/s]



--- Phase 3: Populating Subset Data ---


Processing Years: 100%|██████████| 6/6 [00:40<00:00,  6.83s/it]


In [5]:
# =========================================================
# 5. Phase 4: Global Integrity Verification
# =========================================================
print("\n" + "="*80)
print(f"--- Starting Global Integrity Check ---")
print("="*80)

# Verification Config
NUM_TICKERS_TO_CHECK = 15   # How many random tickers to audit at the end
SAMPLES_PER_TICKER = 2      # Random rows to check per ticker

all_tickers = list(ticker_row_counts.keys())
selected_tickers = random.sample(all_tickers, min(NUM_TICKERS_TO_CHECK, len(all_tickers)))

print(f"Checking {len(selected_tickers)} random tickers...")
print("-" * 85)
print(f"{'Ticker':<8} | {'Shape (Rows, Cols)':<20} | {'Sample Time':<22} | {'Feat[0]':<8} | {'Status'}")
print("-" * 85)

for ticker in selected_tickers:
    try:
        feature_path = os.path.join(dest_dir, f"{ticker}.npy")
        time_path = os.path.join(dest_dir, f"{ticker}_time.npy")
        
        if not os.path.exists(feature_path) or not os.path.exists(time_path):
            print(f"{ticker:<8} | {'MISSING FILES':<20} | {'---':<22} | {'---':<8} | FAIL")
            continue

        # Load via Memory Map
        features = np.load(feature_path, mmap_mode='r')
        times = np.load(time_path, mmap_mode='r')
        
        # Consistency Check
        if features.shape[0] != times.shape[0]:
            print(f"{ticker:<8} | {str(features.shape):<20} | {'MISMATCH':<22} | {'---':<8} | FAIL (Shape)")
            continue
            
        # Random Sampling Check
        for _ in range(SAMPLES_PER_TICKER):
            idx = np.random.randint(0, len(features))
            row_feats = features[idx]
            
            # Decode Timestamp
            ts = pd.to_datetime(times[idx], unit='ns')
            
            # Logic Check: Ensure it falls within our 2007-2012 subset
            if ts.year < int(target_years[0]) or ts.year > int(target_years[-1]):
                status = f"FAIL (Year {ts.year})"
            else:
                status = "OK"

            print(f"{ticker:<8} | {str(features.shape):<20} | {str(ts):<22} | {row_feats[0]:<8.3f} | {status}")
            
    except Exception as e:
        print(f"{ticker:<8} | {'ERROR':<20} | {str(e)[:20]:<22} | {'---':<8} | CRASH")

print("-" * 85)
print("Data Conversion and Verification Complete.")


--- Starting Global Integrity Check ---
Checking 15 random tickers...
-------------------------------------------------------------------------------------
Ticker   | Shape (Rows, Cols)   | Sample Time            | Feat[0]  | Status
-------------------------------------------------------------------------------------
BP       | (118674, 20)         | 2012-07-26 14:20:00    | -0.462   | OK
BP       | (118674, 20)         | 2010-11-18 10:30:00    | 0.997    | OK
BUD      | (39496, 20)          | 2011-04-27 11:30:00    | -1.000   | OK
BUD      | (39496, 20)          | 2011-11-04 12:05:00    | -1.000   | OK
AMRN     | (19647, 20)          | 2012-03-23 09:40:00    | 0.068    | OK
AMRN     | (19647, 20)          | 2012-09-24 10:00:00    | -1.000   | OK
AIV      | (48499, 20)          | 2012-10-08 15:00:00    | 1.000    | OK
AIV      | (48499, 20)          | 2010-05-11 15:50:00    | -1.000   | OK
PVH      | (98572, 20)          | 2010-02-04 15:50:00    | -1.000   | OK
PVH      | (98572, 20) 